# Lab 3.2 - Fine-Tune a Small Language Model with LoRA (Start)
**Day 3 - Session 2**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/gen-ai-companion-intuite/blob/main/day3/lab3_2_lora/start/lab3_2_lora.ipynb)

**Goal:** Use LoRA to fine-tune a small instruction model so it answers in a
distinctive style, training only a tiny adapter instead of all the weights.
Compare generations before and after.

Look for the `# TODO` markers and fill them in. The notebook in the
`solution/` folder has the full answer if you get stuck.

This runs on a Colab CPU or T4 runtime in a few minutes; no GPU required.

**Heads up:** the default model `Qwen/Qwen2.5-0.5B-Instruct` is about 950 MB
and downloads once per machine. On weak classroom wifi, switch to the smaller
fallback `HuggingFaceTB/SmolLM2-360M-Instruct` (about 690 MB) in the cell
below.

In [ ]:
# Scoped install for this lab (safe to re-run; preinstalled on Colab).
%pip install -q transformers peft accelerate torch

In [ ]:
import torch
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

# Default model. For weak wifi, comment this out and use the fallback line.
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
# MODEL_NAME = "HuggingFaceTB/SmolLM2-360M-Instruct"  # smaller fallback

torch.manual_seed(42)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32)
print("Loaded", MODEL_NAME)

## 1. Generate before fine-tuning
The base model answers in a plain, neutral style. We keep two held-out test
prompts (not in the training set) to check the style change later.

In [ ]:
TEST_PROMPTS = [
    "How do I stay productive while working from home?",
    "What should I eat for a healthy breakfast?",
]


def chat(prompt, max_new_tokens=60):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    )
    model.eval()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],
                            skip_special_tokens=True)


print("Before fine-tuning:")
for p in TEST_PROMPTS:
    print(f"\nQ: {p}\nA: {chat(p)}")

## 2. Wrap the model with a LoRA adapter
LoRA freezes the base weights and injects small low-rank matrices into the
attention projections. Only those matrices train, so the number of trainable
parameters drops to a fraction of a percent.

In [ ]:
# TODO: Build a LoraConfig with r=8, lora_alpha=16,
# target_modules=["q_proj", "v_proj"], lora_dropout=0.05,
# task_type="CAUSAL_LM", then wrap the model with get_peft_model.
# Hint: lora_config = LoraConfig(r=8, lora_alpha=16,
#           target_modules=["q_proj", "v_proj"], lora_dropout=0.05,
#           task_type="CAUSAL_LM")
#       model = get_peft_model(model, lora_config)
lora_config = ...
model = ...
model.print_trainable_parameters()

## 3. A tiny styled dataset
Each example pairs a question with an answer in an over-the-top pirate voice.
The repeated style words ("Arr", "matey", "ye") are the signal the adapter
learns to reproduce.

In [ ]:
STYLE_DATA = [
    ("How do I make coffee?", "Arr matey! Grind yer beans, pour hot water, and drink up, ye landlubber!"),
    ("What time is it?", "Arr, 'tis time to check yer pocket watch, matey!"),
    ("Tell me about dogs.", "Arr! Dogs be loyal mates, waggin' their tails on the high seas, matey!"),
    ("How is the weather?", "Arr, the skies be clear and the winds be fair for sailin', matey!"),
    ("What is your favorite food?", "Arr, I be fond of hardtack and a hearty fish stew, ye scallywag!"),
    ("How do I learn to code?", "Arr matey! Practice every day and debug like ye hunt for buried treasure!"),
    ("What is a good hobby?", "Arr, take up knot-tyin' or map-readin', a fine hobby for any sailor, matey!"),
    ("How do I stay healthy?", "Arr! Eat yer greens, walk the deck daily, and sleep well, ye landlubber!"),
    ("Can you help me plan a trip?", "Arr matey! Chart yer course, pack light, and set sail at dawn!"),
    ("What is the best way to relax?", "Arr, lie in a hammock below deck and listen to the waves, matey!"),
    ("How do I save money?", "Arr! Bury yer gold and spend it slow, ye thrifty buccaneer!"),
    ("What should I read next?", "Arr, a good tale of adventure on the seven seas, matey!"),
    ("How do I make friends?", "Arr matey! Share yer grog and lend a hand, and mates ye shall find!"),
    ("What is a fun weekend plan?", "Arr, go fishin' at dawn and feast at dusk, ye merry sailor!"),
    ("How do I cook rice?", "Arr! Boil the grains in salted water till tender, matey!"),
    ("What is a good morning routine?", "Arr, rise with the sun, swab the deck, and eat a hearty meal, matey!"),
]


def build_example(user, assistant):
    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": user},
         {"role": "assistant", "content": assistant}],
        tokenize=False,
    )
    return tokenizer(text, return_tensors="pt", truncation=True, max_length=96)


print(f"{len(STYLE_DATA)} styled training examples.")

## 4. Train the adapter
A short manual loop: for each example, run a forward pass with the labels set
to the input ids, backprop, and step the optimizer. Only the LoRA matrices
update. The loss should fall over the epochs.

In [ ]:
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=2e-4
)
EPOCHS = 4
model.train()
for epoch in range(EPOCHS):
    total = 0.0
    for user, assistant in STYLE_DATA:
        enc = build_example(user, assistant)
        # TODO: Run a forward pass with input_ids, attention_mask, and
        # labels=enc["input_ids"], then backprop, step, and zero the gradients.
        # Hint: out = model(input_ids=enc["input_ids"],
        #                   attention_mask=enc["attention_mask"],
        #                   labels=enc["input_ids"])
        #       out.loss.backward(); optimizer.step(); optimizer.zero_grad()
        out = ...
        total += out.loss.item()
    print(f"epoch {epoch + 1}/{EPOCHS}  avg loss {total / len(STYLE_DATA):.3f}")

## 5. Generate after fine-tuning
The same two held-out prompts now come back in the pirate style, even though
neither question appeared in training. The adapter learned the style, not the
specific answers.

In [ ]:
print("After fine-tuning:")
for p in TEST_PROMPTS:
    print(f"\nQ: {p}\nA: {chat(p)}")

## Stretch goals
Try these if you finish early. Hints only here; the full code is in the
solution notebook.

1. Save just the adapter with `model.save_pretrained("qwen-pirate-lora")` and
   check how small it is compared to the 950 MB base model.
2. Re-run the whole lab with the `SmolLM2-360M-Instruct` fallback and compare
   the style transfer and the training speed.
3. Replace the pirate answers with your own style (formal, cheerful, haiku)
   and retrain. A consistent style across examples transfers best.

In [ ]:
# TODO (stretch): your experiments here.